### Multimodal Model: Gated Fusion 

This notebook establishes our baseline multimodal model for the sexism binary classificaiton.

## Gated Fusion Strategy:
- Text Encoder 
- Img Encoder
- MLP for EEG and ET (Base on our Sensorial Analysis)
- Gated Fusion (Embeddings): $ z = Text + Image + \alpha * ET + \lambda * EEG$


In [1]:
#Libraries
import os
import sys
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm.auto import tqdm
from PIL import Image
from transformers import AutoTokenizer, AutoModel,CLIPModel, CLIPProcessor, get_linear_schedule_with_warmup
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

c:\Users\diego\anaconda3\envs\multimodal_exist\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [3]:
import transformers

In [4]:
print(transformers.__version__)

4.57.0


In [ ]:

# Seeds
SEEDS = [42, 123, 456]

# Fusion strategies to evaluate  ← change this list as needed
FUSION_TYPES = ["mean", "concat"]   # e.g. ["mean", "concat", "weighted_mean", "proj_concat", "gated"]

# Paths
os.chdir("/workspace/dzuniga/multimodal-argmining")
DATA_PATH  = "data/"
IMG_PATH   = "data/images"
OUTPUT_DIR = "results/multimodal/seed_consistency/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Models
TEXT_MODEL_NAME   = "microsoft/deberta-v3-base"
VISION_MODEL_NAME = "openai/clip-vit-base-patch32"

# Architecture
MAX_TEXT_LENGTH = 124
COMMON_DIM      = 768

# Training
BATCH_SIZE    = 16
NUM_EPOCHS    = 15
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 1e-4
WARMUP_RATIO  = 0.1
PATIENCE      = 5
NUM_WORKERS   = 1
PIN_MEMORY    = True if torch.cuda.is_available() else False

print(f"Fusion types : {FUSION_TYPES}")
print(f"Seeds        : {SEEDS}")
print(f"Text model   : {TEXT_MODEL_NAME}")
print(f"Vision model : {VISION_MODEL_NAME}")
print(f"Batch size   : {BATCH_SIZE}  |  Epochs: {NUM_EPOCHS}  |  LR: {LEARNING_RATE}")

## 1. Data Loading

In [ ]:
df = pd.read_csv("data/new_dataset_annotated.csv")

df_train = df[df["split"] == "train"].copy()
df_dev   = df[df["split"] == "dev"].copy()
df_test  = df[df["split"] == "test"].copy()

stance_2id = {"oppose": 0, "support": 1}
for split_df in [df_train, df_dev, df_test]:
    split_df["label"] = split_df["stance"].map(stance_2id)

print(f"Train : {len(df_train)} samples  (oppose={( df_train['label']==0).sum()}, support={(df_train['label']==1).sum()})")
print(f"Dev   : {len(df_dev)}   samples  (oppose={(df_dev['label']==0).sum()}, support={(df_dev['label']==1).sum()})")
print(f"Test  : {len(df_test)}  samples  (oppose={(df_test['label']==0).sum()}, support={(df_test['label']==1).sum()})")

## 2. Tokenizer & Processor (loaded once)

In [ ]:
tokenizer      = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(VISION_MODEL_NAME)
print("Tokenizer and CLIPProcessor loaded.")

## 3. Dataset & DataLoader

In [ ]:
class MultimodalDatasetCLIP(Dataset):
    """
    Dataset for CLIP + DeBERTa multimodal learning.
    Returns tokenized text + processed image (PIL) + label.
    Corrupted images are replaced with a neutral grey fallback.
    """
    def __init__(self, dataframe, img_dir, tokenizer, processor, max_length=124):
        self.df         = dataframe.reset_index(drop=True)
        self.img_dir    = img_dir
        self.tokenizer  = tokenizer
        self.processor  = processor
        self.max_length = max_length

        self.class_counts = self.df['label'].value_counts().to_dict()
        self.num_samples  = len(self.df)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image
        img_path = os.path.join(self.img_dir, str(row['tweet_id']) + ".jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new("RGB", (224, 224), color=(128, 128, 128))

        # Text
        text     = str(row['tweet_text'])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'pixel_values' : image,
            'input_ids'    : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label'        : torch.tensor(row['label'], dtype=torch.long),
            'tweet_id'     : str(row['tweet_id']),
            'text'         : text
        }


def collate_fn_clip(batch, processor):
    images         = [item['pixel_values']   for item in batch]
    labels         = torch.stack([item['label']          for item in batch])
    input_ids      = torch.stack([item['input_ids']      for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    processed      = processor(images=images, return_tensors="pt")
    return {
        'pixel_values'  : processed['pixel_values'],
        'input_ids'     : input_ids,
        'attention_mask': attention_mask,
        'labels'        : labels
    }


def build_dataloaders(seed):
    """Recreate DataLoaders with a fixed worker seed for reproducibility."""
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(seed)

    train_ds = MultimodalDatasetCLIP(df_train, IMG_PATH, tokenizer, clip_processor, MAX_TEXT_LENGTH)
    dev_ds   = MultimodalDatasetCLIP(df_dev,   IMG_PATH, tokenizer, clip_processor, MAX_TEXT_LENGTH)
    test_ds  = MultimodalDatasetCLIP(df_test,  IMG_PATH, tokenizer, clip_processor, MAX_TEXT_LENGTH)

    _collate = lambda batch: collate_fn_clip(batch, clip_processor)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        collate_fn=_collate, worker_init_fn=seed_worker, generator=g
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=_collate
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=_collate
    )
    return train_loader, dev_loader, test_loader


print("Dataset class and helper functions defined.")

## 4. Model Architecture

In [ ]:
class MultimodalBaseline(nn.Module):
    """
    DeBERTa (text) + CLIP ViT (vision) with configurable Early Fusion.
    Supported fusion_type values: 'concat', 'mean', 'weighted_mean', 'proj_concat', 'gated'
    """
    def __init__(
        self,
        text_model_name   = "microsoft/deberta-v3-base",
        vision_model_name = "openai/clip-vit-base-patch32",
        num_classes  = 2,
        freeze_text  = False,
        freeze_vision= True,
        fusion_type  = "concat",
        common_dim   = 768,
        dropout      = 0.1,
    ):
        super().__init__()
        self.fusion_type = fusion_type
        self.common_dim  = common_dim

        # ── Text Encoder (DeBERTa) ──────────────────────────────────────────
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.text_dim     = self.text_encoder.config.hidden_size  # 768

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # ── Vision Encoder (CLIP ViT) ───────────────────────────────────────
        self.clip_model  = CLIPModel.from_pretrained(vision_model_name)
        self.vision_dim  = self.clip_model.config.projection_dim  # 512

        if freeze_vision:
            for p in self.clip_model.vision_model.parameters():
                p.requires_grad = False
            # Unfreeze last ViT block
            for name, p in self.clip_model.vision_model.named_parameters():
                if "encoder.layers.11" in name:
                    p.requires_grad = True

        # ── Projections to common_dim ───────────────────────────────────────
        self.text_projection   = (nn.Identity() if self.text_dim == common_dim
                                  else nn.Linear(self.text_dim, common_dim))
        self.vision_projection = (nn.Identity() if self.vision_dim == common_dim
                                  else nn.Linear(self.vision_dim, common_dim))

        # ── Fusion modules ──────────────────────────────────────────────────
        if fusion_type == "concat":
            self.fusion_layer = nn.Sequential(
                nn.Linear(common_dim * 2, common_dim), nn.ReLU(), nn.Dropout(dropout))

        elif fusion_type == "mean":
            self.fusion_layer = nn.Identity()

        elif fusion_type == "weighted_mean":
            self.text_weight   = nn.Parameter(torch.tensor(0.5))
            self.vision_weight = nn.Parameter(torch.tensor(0.5))
            self.fusion_layer  = nn.Identity()

        elif fusion_type == "gated":
            self.gate         = nn.Sequential(
                nn.Linear(common_dim * 2, common_dim), nn.Sigmoid())
            self.fusion_layer = nn.Identity()

        elif fusion_type == "proj_concat":
            proj_dim          = common_dim // 2
            self.text_proj    = nn.Linear(common_dim, proj_dim)
            self.vision_proj  = nn.Linear(common_dim, proj_dim)
            self.fusion_layer = nn.Sequential(
                nn.Linear(proj_dim * 2, common_dim), nn.ReLU(), nn.Dropout(dropout))
        else:
            raise ValueError(f"Unknown fusion_type: {fusion_type}")

        # ── Classifier ─────────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(common_dim, common_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(common_dim // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text: CLS token
        text_emb   = self.text_projection(
            self.text_encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state[:, 0, :])

        # Vision: CLIP image embedding
        vision_emb = self.vision_projection(
            self.clip_model.get_image_features(pixel_values=pixel_values))

        # Fusion
        if self.fusion_type == "concat":
            fused = self.fusion_layer(torch.cat([text_emb, vision_emb], dim=1))

        elif self.fusion_type == "mean":
            fused = self.fusion_layer((text_emb + vision_emb) / 2)

        elif self.fusion_type == "weighted_mean":
            w = torch.softmax(torch.stack([self.text_weight, self.vision_weight]), dim=0)
            fused = self.fusion_layer(w[0] * text_emb + w[1] * vision_emb)

        elif self.fusion_type == "gated":
            gate  = self.gate(torch.cat([text_emb, vision_emb], dim=1))
            fused = gate * text_emb + (1 - gate) * vision_emb

        elif self.fusion_type == "proj_concat":
            fused = self.fusion_layer(
                torch.cat([self.text_proj(text_emb), self.vision_proj(vision_emb)], dim=1))

        return self.classifier(fused)


print("MultimodalBaseline architecture defined.")

## 5. Training & Evaluation Functions

In [ ]:
def train_multimodal_model(
    model, train_loader, dev_loader,
    num_epochs=15, learning_rate=2e-5,
    weight_decay=1e-4, warmup_ratio=0.1,
    patience=5, device=DEVICE
):
    model = model.to(device)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    total_steps  = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    criterion  = nn.CrossEntropyLoss()

    print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

    history = {"train_loss": [], "train_f1": [], "dev_loss": [], "dev_f1": [], "learning_rates": []}
    best_f1, best_model_state, epochs_no_improve = 0.0, None, 0

    for epoch in range(num_epochs):
        print(f"\n{'='*60}\nEpoch {epoch+1}/{num_epochs}\n{'='*60}")

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss, train_preds, train_labels = 0.0, [], []

        for batch in tqdm(train_loader, desc="Training"):
            optimizer.zero_grad()
            input_ids       = batch["input_ids"].to(device)
            attention_mask  = batch["attention_mask"].to(device)
            pixel_values    = batch["pixel_values"].to(device)
            labels          = batch["labels"].to(device)

            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask,
                           pixel_values=pixel_values)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item() * labels.size(0)
            train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_f1    = f1_score(train_labels, train_preds, average="binary", pos_label=1, zero_division=0)

        # ── Validation ─────────────────────────────────────────────────────
        model.eval()
        dev_loss, dev_preds, dev_labels = 0.0, [], []

        with torch.no_grad():
            for batch in tqdm(dev_loader, desc="Validation", leave=False):
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pixel_values   = batch["pixel_values"].to(device)
                labels         = batch["labels"].to(device)

                logits = model(input_ids=input_ids,
                               attention_mask=attention_mask,
                               pixel_values=pixel_values)
                loss   = criterion(logits, labels)
                dev_loss += loss.item() * labels.size(0)
                dev_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                dev_labels.extend(labels.cpu().numpy())

        dev_loss /= len(dev_loader.dataset)
        dev_f1    = f1_score(dev_labels, dev_preds, average="binary", pos_label=1, zero_division=0)
        current_lr = scheduler.get_last_lr()[0]

        history["train_loss"].append(train_loss)
        history["train_f1"].append(train_f1)
        history["dev_loss"].append(dev_loss)
        history["dev_f1"].append(dev_f1)
        history["learning_rates"].append(current_lr)

        print(f"TRAIN LOSS: {train_loss:.4f} | F1: {train_f1:.4f}")
        print(f"DEV   LOSS: {dev_loss:.4f} | F1: {dev_f1:.4f}")
        print(f"LR: {current_lr:.2e}")

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            print(f"  ✓ New best Dev F1: {best_f1:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  No improvement for {epochs_no_improve} epoch(s)")
            if epochs_no_improve >= patience:
                print("\n  Early stopping triggered!")
                break

    model.load_state_dict(best_model_state)
    model = model.to(device)
    print(f"\n{'='*60}\nTraining complete! Best Dev F1: {best_f1:.4f}\n{'='*60}")
    return model, history


def evaluate_multimodal_model(model, test_loader, device=DEVICE):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values   = batch["pixel_values"].to(device)
            labels         = batch["labels"].to(device)

            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask,
                           pixel_values=pixel_values)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        "accuracy" : accuracy_score(all_labels, all_preds),
        "f1"       : f1_score(all_labels, all_preds, average="binary", pos_label=1, zero_division=0),
        "precision": precision_score(all_labels, all_preds, pos_label=1, zero_division=0),
        "recall"   : recall_score(all_labels, all_preds, pos_label=1, zero_division=0),
        "report"   : classification_report(all_labels, all_preds, zero_division=0)
    }


print("Training and evaluation functions defined.")

## 6. Seed Helper

In [ ]:
def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"  Seed set to {seed}")

print("set_seed() defined.")

## 7. Experiment Loop

For every combination of **(fusion_type, seed)** we:
1. Set the seed
2. Build fresh DataLoaders with that seed
3. Instantiate the model
4. Train with early stopping
5. Evaluate on the test set
6. Store the metrics

In [ ]:
# raw_results[fusion][seed] = metrics_dict
raw_results = {fusion: {} for fusion in FUSION_TYPES}

total_runs = len(FUSION_TYPES) * len(SEEDS)
run_count  = 0

for fusion in FUSION_TYPES:
    print(f"\n{'#'*80}")
    print(f"# FUSION STRATEGY: {fusion.upper()}")
    print(f"{'#'*80}")

    for seed in SEEDS:
        run_count += 1
        print(f"\n{'─'*60}")
        print(f"  Run {run_count}/{total_runs} | Fusion: {fusion} | Seed: {seed}")
        print(f"{'─'*60}")

        # 1. Fix seed
        set_seed(seed)

        # 2. DataLoaders
        train_loader, dev_loader, test_loader = build_dataloaders(seed)

        # 3. Model
        model = MultimodalBaseline(
            text_model_name   = TEXT_MODEL_NAME,
            vision_model_name = VISION_MODEL_NAME,
            num_classes  = 2,
            freeze_text  = False,
            freeze_vision= True,
            fusion_type  = fusion,
            common_dim   = COMMON_DIM
        )

        # 4. Train
        model, history = train_multimodal_model(
            model         = model,
            train_loader  = train_loader,
            dev_loader    = dev_loader,
            num_epochs    = NUM_EPOCHS,
            learning_rate = LEARNING_RATE,
            weight_decay  = WEIGHT_DECAY,
            warmup_ratio  = WARMUP_RATIO,
            patience      = PATIENCE,
            device        = DEVICE
        )

        # 5. Evaluate
        print("\nEvaluating on TEST set...")
        test_metrics = evaluate_multimodal_model(model, test_loader, device=DEVICE)

        # 6. Store
        raw_results[fusion][seed] = test_metrics

        print(f"\n  TEST | Accuracy: {test_metrics['accuracy']:.4f} "
              f"| F1: {test_metrics['f1']:.4f} "
              f"| Precision: {test_metrics['precision']:.4f} "
              f"| Recall: {test_metrics['recall']:.4f}")

        # Free GPU memory
        del model
        torch.cuda.empty_cache()

print("\n" + "="*80)
print("ALL RUNS COMPLETE")
print("="*80)

## 8. Results: Mean ± Std across Seeds

In [ ]:
# ─── Build summary DataFrame ──────────────────────────────────────────────────
metric_keys = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = [
    ('Accuracy',  'Accuracy_Mean',  'Accuracy_Std'),
    ('Precision', 'Precision_Mean', 'Precision_Std'),
    ('Recall',    'Recall_Mean',    'Recall_Std'),
    ('F1',        'F1_Mean',        'F1_Std'),
]

rows = []
for fusion in FUSION_TYPES:
    row = {"Fusion": fusion}

    for metric_key, (_, mean_col, std_col) in zip(metric_keys, metric_labels):
        values = [raw_results[fusion][s][metric_key] for s in SEEDS]
        row[mean_col] = round(np.mean(values), 4)
        row[std_col]  = round(np.std(values),  4)

        # Also store per-seed values for transparency
        for s in SEEDS:
            row[f"{metric_key}_seed{s}"] = round(raw_results[fusion][s][metric_key], 4)

    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("F1_Mean", ascending=False).reset_index(drop=True)

# ─── Display: compact view (Mean ± Std only) ──────────────────────────────────
compact_cols = ["Fusion"]
for _, mean_col, std_col in metric_labels:
    compact_cols += [mean_col, std_col]

print("\n" + "="*80)
print("SEED CONSISTENCY RESULTS  (sorted by F1_Mean ↓)")
print("="*80)
display(summary_df[compact_cols])

In [ ]:
# ─── Per-seed breakdown ───────────────────────────────────────────────────────
per_seed_cols = ["Fusion"]
for metric_key in metric_keys:
    for s in SEEDS:
        per_seed_cols.append(f"{metric_key}_seed{s}")

print("\nPer-seed breakdown:")
display(summary_df[per_seed_cols])

## 9. Save Results

In [ ]:
# Save full summary
summary_path = os.path.join(OUTPUT_DIR, "seed_consistency_results.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Full results saved to: {summary_path}")

# Save compact (Mean ± Std) version
compact_path = os.path.join(OUTPUT_DIR, "seed_consistency_summary.csv")
summary_df[compact_cols].to_csv(compact_path, index=False)
print(f"Compact summary saved to: {compact_path}")

# Preview
summary_df[compact_cols]

## 10. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Seed Consistency: DeBERTa + CLIP Early Fusion", fontsize=14, fontweight="bold")

x = np.arange(len(FUSION_TYPES))
width = 0.35

for ax, (metric_key, (_, mean_col, std_col)) in zip(
        axes, [(mk, ml) for mk, ml in zip(metric_keys[3:4] + metric_keys[:1],
                                           metric_labels[3:4] + metric_labels[:1])]):
    means = summary_df.sort_values("Fusion")[mean_col].values
    stds  = summary_df.sort_values("Fusion")[std_col].values
    fusions_sorted = summary_df.sort_values("Fusion")["Fusion"].values

    bars = ax.bar(fusions_sorted, means, yerr=stds, capsize=6,
                  color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_title(f"{mean_col.replace('_Mean', '')} (Mean ± Std)")
    ax.set_ylim(max(0, min(means) - 0.05), min(1.0, max(means) + 0.05))
    ax.set_ylabel("Score")
    ax.set_xlabel("Fusion Type")

    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + std + 0.002,
                f"{mean:.3f}\n±{std:.3f}",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "seed_consistency_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to: {plot_path}")